## Generating the training dataset
a a a a b b b EOS, a a b b EOS, a b UNK a b b EOS, etc.

In [7]:
import numpy as np

np.random.seed(42)

def generate_dataset(num_sequence=100):
    """
    Generates a number of sequences as our dataset.
    
    Args:
     `num_sequences`: the number of sequences to be generated.
     
    Returns a list of sequences.
    """
    samples = []
    
    for _ in range(num_sequence):
        num_tokens = np.random.randint(1,10)
        sample = ['a'] * num_tokens + ['b'] * num_tokens + ['EOS']
        samples.append(sample)
    return samples
# print(generate_dataset())
sequences = generate_dataset()

In [8]:
from collections import defaultdict

def sequences_to_dicts(sequences):
    """
    Creates a word_to_idx and idx_to_word dictionaries for a list of sequences.
    """
    flatten = lambda l: [item for sublist in l for item in sublist]
    
    # Flatten the dataset
    all_words = flatten(sequences)
    
    # Count the number of word occurences
    word_count = defaultdict(int)
    for word in all_words:
        word_count[word] += 1
    
    print(word_count)
    
    # Sort by frequency
    word_count = sorted(list(word_count.items()),key = lambda l: -l[1])
    print(word_count)
    
    # Create a list of unique words
    unique_words = [item[0] for item in word_count]
    print(unique_words)
    
    # Adding UNK to the list of unique_words
    unique_words.append('UNK')
    
    # Count number of sequences and number of unique words
    num_sentences, vocab_size = len(sequences), len(unique_words)
    print(num_sentences, vocab_size)
    # Create dictionaries so that we can go from word to index and back
    # If a word is not in our vocabulary, we assign it to token 'UNK'
    word_to_idx = defaultdict(lambda: num_words)
    idx_to_word = defaultdict(lambda: 'UNK')    
    
    # Fill dictionaries
    for idx, word in enumerate(unique_words):
        # YOUR CODE HERE!
        word_to_idx[word] = idx
        idx_to_word[idx] = word
    print(word_to_idx)
    print(idx_to_word)
    
    return word_to_idx, idx_to_word, num_sentences, vocab_size
# sequences_to_dicts(sequences)

word_to_idx, idx_to_word, num_sequences, vocab_size = sequences_to_dicts(sequences)

print(f'We have {num_sequences} sentences and {len(word_to_idx)} unique tokens in our dataset (including UNK).\n')
print('The index of \'b\' is', word_to_idx['b'])
print(f'The word corresponding to index 1 is \'{idx_to_word[1]}\'')

defaultdict(<class 'int'>, {'a': 532, 'b': 532, 'EOS': 100})
[('a', 532), ('b', 532), ('EOS', 100)]
['a', 'b', 'EOS']
100 4
defaultdict(<function sequences_to_dicts.<locals>.<lambda> at 0x000001E5933145E0>, {'a': 0, 'b': 1, 'EOS': 2, 'UNK': 3})
defaultdict(<function sequences_to_dicts.<locals>.<lambda> at 0x000001E593314AE0>, {0: 'a', 1: 'b', 2: 'EOS', 3: 'UNK'})
We have 100 sentences and 4 unique tokens in our dataset (including UNK).

The index of 'b' is 1
The word corresponding to index 1 is 'b'


In [ ]:
# !jupyter kernelspec list

In [ ]:
# %pip install torch

## Partioning the dataset

Our dataset needs inputs and targets for each sequences. Further, we need to partition our input examples i.e., sentences (since we are trying to implement next word predictor) into Train, Validation and Test sets. For the next word predictor, our target sequence is input sequence shifted by one word. 

Will use PyTorch Dataset class to build simple dataset from which we can easily retrieve (inputs,targets) pair for each of the sequences. 

In [9]:
from torch.utils import data

class Dataset(data.Dataset):
    def __init__(self,inputs,targets):
        self.inputs = inputs
        self.targets = targets
        
    def __len__(self):
        # Return size of dataset
        return len(self.targets)
    
    def __getitem__(self,index):
        # Retrieve inputs and targets at the given index
        X = self.inputs[index]
        y = self.targets[index]
        
        return X,y

In [12]:
def create_datasets(sequences, dataset_class, p_train = 0.8, p_val= 0.1, p_test= 0.1):
    # Define partion size
    num_train = int(len(sequences)*p_train)
    num_val = int(len(sequences)*p_val)
    num_test = int(len(sequences)*p_test)
    
    # Split sequences into partitions
    sequences_train = sequences[:num_train]
    sequences_val = sequences[num_train:num_train + num_val]
    sequences_test = sequences[-num_test:]
    
    def get_inputs_targets_from_sequences(sequences):
        inputs, targets = [], []
        
        for sequence in sequences:
            # print("This is sequence: ",sequence)
            inputs.append(sequence[:-1])
            targets.append(sequence[1:])
        
        return inputs, targets 
    
    # Get inputs and targets for each partition
    inputs_train, targets_train = get_inputs_targets_from_sequences(sequences_train)
    inputs_val, targets_val = get_inputs_targets_from_sequences(sequences_val)
    inputs_test, targets_test = get_inputs_targets_from_sequences(sequences_test)
    
    # Create datasets for the inputs and targets creater above
    training_set = dataset_class(inputs_train, targets_train)
    validation_set = dataset_class(inputs_val, targets_val)
    test_set = dataset_class(inputs_test, targets_test)
    
    return training_set, validation_set, test_set

training_set, validation_set, test_set = create_datasets(sequences, Dataset)

print(f'We have {len(training_set)} samples in the training set.')
print(f'We have {len(validation_set)} samples in the validation set.')
print(f'We have {len(test_set)} samples in the test set.')

We have 80 samples in the training set.
We have 10 samples in the validation set.
We have 10 samples in the test set.


## One-hot encodings

In [22]:
def one_hot_encode(idx, vocab_size):
    one_hot = np.zeros(vocab_size)
    
    # set teh appropriate element to one
    one_hot[idx] = 1.0
    
    return one_hot

def one_hot_encode_sequence(sequence, vocab_size):
    
    # encode each word in the sentence
    encoding = np.array([one_hot_encode(word_to_idx[word], vocab_size) for word in sequence])
    
    # Reshaping encoding s.t. it has shape (num_words, vocab_size, 1)
    print("Shape of encoding: ",encoding.shape)
    encoding = encoding.reshape(encoding.shape[0], encoding.shape[1],1)
    
    return encoding

test_word = one_hot_encode(word_to_idx['a'], vocab_size)
print(f'Our one-hot encoding of \'a\' has shape {test_word.shape}.')

test_sentence = one_hot_encode_sequence(['a', 'b'], vocab_size)
print(f'Our one-hot encoding of \'a b\' has shape {test_sentence.shape}.')
print("test_sentence encoding: \n",test_sentence)
    

Our one-hot encoding of 'a' has shape (4,).
Shape of encoding:  (2, 4)
Our one-hot encoding of 'a b' has shape (2, 4, 1).
test_sentence encoding: 
 [[[1.]
  [0.]
  [0.]
  [0.]]

 [[0.]
  [1.]
  [0.]
  [0.]]]


# Long Short-Term Memory (LSTM) Cell

## Initialization of a LSTM network

In [25]:
hidden_size = 50 # Number of dimensions in the hidden state
vocab_size  = len(word_to_idx) # Size of the vocabulary used

def init_orthogonal(param):
    """
    Initializes weight parameters orthogonally.
    
    Refer to this paper for an explanation of this initialization:
    https://arxiv.org/abs/1312.6120
    """
    if param.ndim < 2:
        raise ValueError("Only parameters with 2 or more dimensions are supported.")

    rows, cols = param.shape
    
    new_param = np.random.randn(rows, cols)
    
    if rows < cols:
        new_param = new_param.T
    
    # Compute QR factorization
    q, r = np.linalg.qr(new_param)
    
    # Make Q uniform according to https://arxiv.org/pdf/math-ph/0609050.pdf
    d = np.diag(r, 0)
    ph = np.sign(d)
    q *= ph

    if rows < cols:
        q = q.T
    
    new_param = q
    
    return new_param

In [31]:
# Size of concatenated hidden + vocab_size
z_size = hidden_size + vocab_size

def init_lstm(hidden_size, vocab_size, z_size):
    
    # Weight matrix (forget gate)    
    W_f = np.random.randn(hidden_size, z_size)
    print("Shape of W_f: ",W_f.shape)
    
    # Bias for forget gate
    b_f = np.zeros((hidden_size, 1))
    
    # Weight matrix (input gate) 
    W_i = np.random.randn(hidden_size, z_size)
    
    # Bias for input gate
    b_i = np.zeros((hidden_size, 1))
    
    # Weight matrix (candidate) 
    W_g = np.random.randn(hidden_size, z_size)
    
    # Bias for candidate
    b_g = np.zeros((hidden_size, 1))
    
    # Weight matrix (output gate) 
    W_o = np.random.randn(hidden_size, z_size)
    
    # Bias for input gate
    b_o = np.zeros((hidden_size, 1))
    
    # Weight matrix relating the hidden-state to the output
    W_v = np.random.randn(vocab_size, hidden_size)
    b_v = np.zeros((vocab_size, 1))
    print("Shape of W_v: ",W_v.shape)
    
    # Initializing weights
    W_f = init_orthogonal(W_f)
    W_i = init_orthogonal(W_i)
    W_g = init_orthogonal(W_g)
    W_o = init_orthogonal(W_o)
    W_v = init_orthogonal(W_v)
    
    return W_f, W_i, W_g, W_o, W_v

params = init_lstm(hidden_size=hidden_size, vocab_size=vocab_size, z_size=z_size)

Shape of W_f:  (50, 54)
Shape of W_v:  (4, 50)


## Forward Pass

In [ ]:
def forward(inputs, h_prev, C_prev, p):
    """
    Arguments:
    x -- your input data at timestep "t", numpy array of shape (n_x, m).
    h_prev -- Hidden state at timestep "t-1", numpy array of shape (n_a, m)
    C_prev -- Memory state at timestep "t-1", numpy array of shape (n_a, m)
    p -- python list containing:
                        W_f -- Weight matrix of the forget gate, numpy array of shape (n_a, n_a + n_x)
                        b_f -- Bias of the forget gate, numpy array of shape (n_a, 1)
                        W_i -- Weight matrix of the update gate, numpy array of shape (n_a, n_a + n_x)
                        b_i -- Bias of the update gate, numpy array of shape (n_a, 1)
                        W_g -- Weight matrix of the first "tanh", numpy array of shape (n_a, n_a + n_x)
                        b_g --  Bias of the first "tanh", numpy array of shape (n_a, 1)
                        W_o -- Weight matrix of the output gate, numpy array of shape (n_a, n_a + n_x)
                        b_o --  Bias of the output gate, numpy array of shape (n_a, 1)
                        W_v -- Weight matrix relating the hidden-state to the output, numpy array of shape (n_v, n_a)
                        b_v -- Bias relating the hidden-state to the output, numpy array of shape (n_v, 1)
    Returns:
    z_s, f_s, i_s, g_s, C_s, o_s, h_s, v_s -- lists of size m containing the computations in each forward pass
    outputs -- prediction at timestep "t", numpy array of shape (n_v, m)
    """
    assert h_prev.shape == (hidden_size, 1)
    assert C_prev.shape == (hidden_size, 1)
    
    # unpacking our parameters
    W_f,, W_i, W_g, W_o, W_v, b_f, b_i, b_g, b_o, b_v = p
    
    # Save a list of computations for each of the component in the LSTM
    x_s, z_s, f_s, i_s,  = [], [] ,[], []
    g_s, C_s, o_s, h_s = [], [] ,[], []
    v_s, output_s =  [], [] 
    
    # Append the initial cell and hidden state to their respective lists
    h_s.append(h_prev)
    C_s.append(C_prev)
    
    for x in inputs:
        # Concatenate input and hiddne state
        z = np.row_stack((h_prev, x))
        z_s.append(z)
        
        # Calculate forget gate
        f = sigmoid(np.dot(W_f,z) + b_f)
        f_s.append(f)
        
        # Calculate input gate
        i = sigmoid(np.dot(W_i,z) + b_i)
        i_s.append(i)
        
        # Calculate candidate
        g = tanh(np.dot(W_g, z) + b_g)
        g_s.append(g)
        
        # Calculate memory state
        C_prev = f * C_prev + i * g
        C_s.append(C_prev)
        
        # Calculate output gate
        o = sigmoid(np.dot(W_o,z) + b_o)
        o_s.append(o)
        
        # Calculate hiddent state
        h_prev = o * tanh(C_prev)
        h_s.append(h_prev)
        
        # Calculate logits
        v = np.dot(W_v, h_prev) + b_v
        v_s.append(v)
        
        # Calculate softmax
        output = softmax(v)
        output_s.append(output)
        
    return z_s, f_s, i_s, g_s, C_s, o_s, h_s, v_s, output_s

# Get first sentence in test set
inputs, targets = test_set[1]

# One hot encode input and target sequence
inputs_one_hot = one_hot_encode_sequence(inputs, vocab_size)
targets_one_hot = one_hot_encode_sequence(targets, vocab_size)

# initialize hidden state as zeros
h = np.zeros((hidden_size, 1))
c = np.zeros((hidden_size, 1))

# Forward pass
z_s, f_s, i_s, g_s, C_s, o_s, h_s, v_s, outputs = forward(inputs_one_hot, h, c, params)